# 02. BdG matrix, eigenvalues and spectrum plots

Follow the matrix assembly from its smaller blocks to Eigensystem, then read the energy-spectrum diagrams.

Read each explanation together with the code directly below it. The code is original Mathematica; the `.nb` files are the executable versions. Diagrams are existing saved results. These reading notebooks do not rerun the calculation.

## 1. Build the onsite pairing block

The nested loops construct `js`: for every site `i`, they find the site with the same `xpos` and `ypos`. With unique coordinates, `js[[i]]` contains `i` itself. This is why the pairing update is onsite.

`st` stores one pairing value per site. `Flatten[st]` turns its nested list into a vector. `SCs2[st]` places that vector on the diagonal. `SCs1` is a zero block. `SCs[st]` places `SCs2[st]` in the two off-diagonal spin blocks, giving a `2 NN × 2 NN` pairing matrix. `SCs3` and the helper `s[i_,j_]` are defined here but are not used in the active iteration shown next.

**Onsite pairing definitions** — `BdG topological.nb`, input 2.

```mathematica
(*------onsite------*)
Clear[s,st,stab,sij,SCs,dmn,dmm]

js={};
For[i=1,i<=NN,i++,jisel={};
For[j=1,j<=NN,j++,
If[xpos[i]==xpos[j],
If[ypos[i]==ypos[j],AppendTo[jisel,j]]]];AppendTo[js,jisel]]



SCs1=SparseArray[{x_,y_}/;Not[TrueQ[sij[x][y]==Null]]\[Rule]0,{NN,NN}];
SCs2[st_]:= SparseArray[{Band[{1,1}]\[Rule] Flatten[st]}, {NN, NN}];
SCs3=SparseArray[{x_,y_}/;Not[TrueQ[sij[x][y]==Null]]\[Rule]0,{NN,NN}];
SCs[st_]:= ArrayFlatten[{{SCs1, SCs2[st]}, {SCs2[st], SCs1}}];
Clear[sij]
s[i_,j_]:=Take[st[[i,j]]];
```

## 2. Assemble the full matrix

For `NN=400`, each spin block is 400 × 400, `hh1` and `hh2` are 800 × 800, and `H` is 1600 × 1600.

| Variable | What the active code puts into it |
| --- | --- |
| `Huu`, `Hdd` | Same-spin hopping plus the diagonal chemical-potential shift. Both use the same expression here. |
| `Hud` | The spin-mixing matrix `soc`. |
| `Hdu` | The complex conjugate of `soc`. |
| `hh1` | The four electron spin blocks joined into one matrix. |
| `hh2` | The corresponding hole blocks with the signs and conjugations displayed in the code. |
| `Huv` | The pairing block `SCs[st]`. |
| `Hvu` | Its complex conjugate. |
| `H` | The complete electron–hole block matrix. |

`ArrayFlatten` joins blocks; it does not diagonalize them. `Conjugate` reverses the sign of the imaginary part. The exact signs are retained below.

**Matrix assembly and diagonalization** — `BdG topological.nb`, input 5.

```mathematica
  Huu = H1-Chempot1[mu] (*+imp*)(**+Hartree*);
  Hdd = H1-Chempot1[mu];
Hud = soc;
Hdu = Conjugate[soc];
hh1 = ArrayFlatten[{{Huu,Hud}, {Hdu, Hdd}}];
hh2 = ArrayFlatten[{{-Conjugate[Huu],Conjugate[Hud]}, {Conjugate[Hdu], -Conjugate[Hdd]}}];
    Huv= SCs[st]; (*Superconducting part \[CapitalDelta]*)
  Hvu=Conjugate[Huv];
  H = ArrayFlatten[{{hh1, Huv}, {Hvu, hh2}}]; (*Joins all the matrices*)
 (* Print[HermitianMatrixQ[H]];*)
  (*Print[MatrixForm[H]];*)
  
  ES = Eigensystem[H];
 

```

## 3. What Eigensystem returns

`ES = Eigensystem[H]` returns two lists. `ES[[1]]` contains the eigenvalues, or energies. `ES[[2]]` contains the corresponding eigenvectors, stored as a list of vectors. Each vector has `4 NN` components.

The calculation solves `H . vector = energy vector`. An eigenvalue says where a level lies in energy. Its eigenvector says how the state is distributed across sites and component groups. Plotting the eigenvalues alone does not display that spatial distribution.

In the later occupation sums, `ES[[2]][[;;,i]]` selects component `i` from every eigenvector. The offsets `i+NN`, `i+2NN` and `i+3NN` select the same site in the other three component groups.

## 4. How ListPlot draws the spectrum

`Sort[ES[[1]]]` puts the energies in ascending order. Since `ListPlot` receives values rather than explicit coordinate pairs, it uses their positions in this sorted list as the horizontal coordinate. The horizontal `n` is therefore a **state index**, not a lattice position or momentum.

`PlotRange` selects the visible energy interval. `Frame` draws a frame; `FrameLabel` labels its axes. `GridLines` places horizontal reference lines at the specified energies, and `Directive[Green,Dashed]` styles them. The lines are chosen plotting parameters, not automatically detected gap edges. `PlotLabel` is saved text; it is not updated automatically from the current values of `mu` or `st`.

**Spectrum plotting command** — `BdG topological.nb`, input 7.

```mathematica
ListPlot[Sort[ES[[1]]],PlotRange\[Rule]{-0.6,0.6} ,Frame\[Rule]True,PlotLabel\[Rule]"\<\!\(\*SubscriptBox[\(\[CapitalDelta]\), \(o\)]\)=\[PlusMinus]0.7, M=0, m=0.5, \[Lambda]=1, <n>=1, \!\(\*SubscriptBox[\(\[CapitalDelta]\), \(con\)]\)=0.064, \!\(\*SubscriptBox[\(\[Mu]\), \(con\)]\)=-0.46 \>",FrameLabel\[Rule]{n,Energy},LabelStyle\[Rule]"\<DisplayFormula\>",GridLines\[Rule]{None,{-0.05,0.05}},GridLinesStyle\[Rule]Directive[Green, Dashed], Frame\[Rule]True, FrameStyle\[Rule]Gray,ImageSize\[Rule]Large]
```

**Second saved plotting command** — `added vortex.nb`, input 7.

```mathematica
ListPlot[Sort[ES[[1]]],PlotRange\[Rule]{-0.6,0.6} ,Frame\[Rule]True,PlotLabel\[Rule]"\<\!\(\*SubscriptBox[\(\[CapitalDelta]\), \(o\)]\)=\[PlusMinus]0.20, m=0.5, \[Lambda]=1, <n>=1, \[Mu]=-0.44 \>",FrameLabel\[Rule]{n,Energy},LabelStyle\[Rule]"\<DisplayFormula\>",GridLines\[Rule]{None,{-0.20,0.20}},GridLinesStyle\[Rule]Directive[Green, Dashed], Frame\[Rule]True, FrameStyle\[Rule]Gray,ImageSize\[Rule]Large]
```

## 5. Read the saved spectrum diagrams

In each diagram, blue points are energy levels and the vertical axis is energy in the displayed units. Several levels may overlap at the resolution of the image. These are saved examples, so their labels and visible ranges may differ from the plotting cells above.

### Spectrum with levels near zero

![Spectrum with levels near zero](https://raw.githubusercontent.com/adicos0/BdG-code-for-Topological-superconductor-with-vortex-/main/diagrams/spectrum_zero_pairing.png)

The title sets the pairing parameter to zero. The blue levels approach and cross the horizontal zero-energy reference without the wide empty central interval seen in the next diagram. The compressed horizontal spread comes from showing only part of the energy range while keeping a broad state-index axis.

[Enlarge the diagram](https://raw.githubusercontent.com/adicos0/BdG-code-for-Topological-superconductor-with-vortex-/main/diagrams/spectrum_zero_pairing.png)

### Spectrum with an empty central interval

![Spectrum with an empty central interval](https://raw.githubusercontent.com/adicos0/BdG-code-for-Topological-superconductor-with-vortex-/main/diagrams/spectrum_gap.png)

The green dashed lines mark approximately +0.06 and -0.06 on the energy axis. The visible points form negative- and positive-energy groups, leaving a central interval with no resolved dots. This is how a gap appears in a sorted-energy plot; its horizontal width is not a physical length.

[Enlarge the diagram](https://raw.githubusercontent.com/adicos0/BdG-code-for-Topological-superconductor-with-vortex-/main/diagrams/spectrum_gap.png)

### Spectrum with a level close to zero

![Spectrum with a level close to zero](https://raw.githubusercontent.com/adicos0/BdG-code-for-Topological-superconductor-with-vortex-/main/diagrams/spectrum_central_state.png)

A blue level appears close to zero energy, between the green reference lines. Other levels lie close to the two reference boundaries. This image gives their energies; it does not show whether their wavefunctions are at the centre or the edge of the lattice, and does not by itself identify the type of state.

[Enlarge the diagram](https://raw.githubusercontent.com/adicos0/BdG-code-for-Topological-superconductor-with-vortex-/main/diagrams/spectrum_central_state.png)